Pulls VisDrone images and real bounding boxes from Hugging Face and lands them in raw.
Uses banu4prasad/VisDrone-Dataset (YOLO format, real boxes) instead of the
FiftyOne-packaged version, which turned out to have no usable metadata -- see README.

Note: these are static images, not real video frames (VisDrone-VID is Google-Drive-only,
no direct link -- see README). scene_id/frame_number below are SYNTHETIC, grouped by us
every 10 images, just so there's something to build the fragment-index pattern on later.

Standard library imports for byte buffers and JSON encoding.

In [1]:
import io
import json

Imports for S3/RustFS access, DuckDB, downloading files from the Hugging Face Hub, and image handling.

In [2]:
import boto3
import duckdb
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Configuration: how many images to pull, the source HF repo/split, and the synthetic clip size used to group frames.

In [3]:
N_IMAGES = 60
BUCKET = "lakehouse"
S3_PREFIX = "assets/visdrone/images"
REPO_ID = "banu4prasad/VisDrone-Dataset"
SPLIT_DIR = "VisDrone2019-DET-val"   # smallest split, 548 images
FRAMES_PER_CLIP = 10                  # synthetic clip size, see note above

VisDrone's 11 object categories, indexed by `class_id` from the YOLO label files.

In [4]:
CLASS_NAMES = [
    "pedestrian", "people", "bicycle", "car", "van", "truck",
    "tricycle", "awning-tricycle", "bus", "motor", "others",
]

Creates a boto3 S3 client pointed at RustFS, same as the COCO script.

In [5]:
# connects to RustFS the same way the COCO script does
def make_s3_client():
    return boto3.client(
        "s3",
        endpoint_url="http://rustfs:9000",
        aws_access_key_id="rustfsadmin",
        aws_secret_access_key="rustfsadmin",
    )

Connects DuckDB and attaches the DuckLake catalog.

In [6]:
# turns on DuckLake, connects it to RustFS, opens the catalog
def attach_lakehouse():
    con = duckdb.connect()
    con.execute(open("sql/00_attach.sql").read())
    return con

Lists the image files in the target split directly from the HF repo, since it has no dataset script or auto Parquet conversion.

In [7]:
# this repo has no dataset script or parquet conversion, so we just list
# the raw files ourselves and filter down to the split/folder we want
def list_image_files(api):
    all_files = api.list_repo_files(REPO_ID, repo_type="dataset")
    images = sorted(
        f for f in all_files
        if f.startswith(f"{SPLIT_DIR}/images/") and f.endswith(".jpg")
    )
    return images[:N_IMAGES]

Parses a YOLO-format label file into a list of real detections (class + normalized bbox).

In [8]:
# YOLO format: one line per object, "class_id x_center y_center width height",
# all normalized 0-1 relative to image size (not pixel coordinates)
def parse_yolo_label(label_path):
    detections = []
    try:
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue   # skip malformed/blank lines instead of crashing
                class_id = int(parts[0])
                x, y, w, h = map(float, parts[1:])
                detections.append({
                    "class_id": class_id,
                    "class_name": CLASS_NAMES[class_id] if class_id < len(CLASS_NAMES) else "unknown",
                    "x_center": x,
                    "y_center": y,
                    "width": w,
                    "height": h,
                })
    except FileNotFoundError:
        pass   # some images just don't have a label file, that's fine
    return detections

Downloads one image and its matching label file, uploads the image to RustFS, and builds its metadata row — including the synthetic scene/frame grouping.

In [9]:
# downloads one image + its matching label file, uploads the image to
# RustFS, and returns a row describing it (with the real detections)
def upload_image_and_build_row(s3, index, image_repo_path):
    local_image_path = hf_hub_download(REPO_ID, image_repo_path, repo_type="dataset")
    img = Image.open(local_image_path).convert("RGB")

    # the matching label file lives in the same position under labels/
    # instead of images/, same filename but .txt instead of .jpg
    label_repo_path = image_repo_path.replace("/images/", "/labels/").replace(".jpg", ".txt")
    try:
        local_label_path = hf_hub_download(REPO_ID, label_repo_path, repo_type="dataset")
        detections = parse_yolo_label(local_label_path)
    except Exception:
        detections = []   # no matching label file found for this image

    key = f"{S3_PREFIX}/{index:04d}.jpg"
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=90)
    buf.seek(0)
    s3.put_object(Bucket=BUCKET, Key=key, Body=buf, ContentType="image/jpeg")

    # synthetic clip grouping -- see the note at the top of this file
    scene_id = f"synthetic-clip-{index // FRAMES_PER_CLIP:03d}"
    frame_number = index % FRAMES_PER_CLIP

    return {
        "image_uri": f"s3://{BUCKET}/{key}",
        "width": img.width,
        "height": img.height,
        "scene_id": scene_id,
        "frame_number": frame_number,
        "n_detections": len(detections),
        "detections_json": json.dumps(detections),
    }

Connects to RustFS, attaches the DuckLake catalog, and creates an HF API client.

In [10]:
s3 = make_s3_client()

con = attach_lakehouse()

api = HfApi()

Finds the target images in the VisDrone HF repo.

In [11]:
print(f"Finding {N_IMAGES} images in {REPO_ID}/{SPLIT_DIR}...")

image_files = list_image_files(api)

print(f"Found {len(image_files)} images to pull")

Finding 60 images in banu4prasad/VisDrone-Dataset/VisDrone2019-DET-val...


Found 60 images to pull


Downloads each image and its label file, uploads the image to RustFS, and collects its metadata row — including the synthetic scene/frame grouping.

In [12]:
rows = []

for i, image_path in enumerate(image_files):
    rows.append(upload_image_and_build_row(s3, i, image_path))

    if (i + 1) % 10 == 0:
        print(f"  ...{i + 1}/{len(image_files)} frames uploaded")

  ...10/60 frames uploaded


  ...20/60 frames uploaded


  ...30/60 frames uploaded


  ...40/60 frames uploaded


  ...50/60 frames uploaded


  ...60/60 frames uploaded


Confirms all frames were uploaded to RustFS.

In [13]:
print(f"Uploaded {len(rows)} frames to s3://{BUCKET}/{S3_PREFIX}/")

Uploaded 60 frames to s3://lakehouse/assets/visdrone/images/


Writes the collected metadata into `raw.visdrone_frames`, creating a new DuckLake snapshot.

In [14]:
df = pd.DataFrame(rows)

con.register("visdrone_df", df)

con.execute("CREATE OR REPLACE TABLE raw.visdrone_frames AS SELECT * FROM visdrone_df")

Confirms the row count and total real detections landed in the table.

In [15]:
count = con.sql("SELECT COUNT(*) FROM raw.visdrone_frames").fetchone()[0]

total_detections = con.sql("SELECT SUM(n_detections) FROM raw.visdrone_frames").fetchone()[0]

print(f"raw.visdrone_frames now has {count} rows, {total_detections} total real detections")

raw.visdrone_frames now has 60 rows, 2789 total real detections


Shows how many frames and detections landed in each synthetic scene.

In [16]:
# frames per synthetic scene -- matters later for the fragment index
print("\nFrames per (synthetic) scene:")

con.sql("SELECT scene_id, COUNT(*) AS n_frames, SUM(n_detections) AS n_detections FROM raw.visdrone_frames GROUP BY scene_id ORDER BY scene_id").show()


Frames per (synthetic) scene:
┌────────────────────┬──────────┬──────────────┐
│      scene_id      │ n_frames │ n_detections │
│      varchar       │  int64   │    int128    │
├────────────────────┼──────────┼──────────────┤
│ synthetic-clip-000 │       10 │         1066 │
│ synthetic-clip-001 │       10 │          229 │
│ synthetic-clip-002 │       10 │          135 │
│ synthetic-clip-003 │       10 │          438 │
│ synthetic-clip-004 │       10 │          598 │
│ synthetic-clip-005 │       10 │          323 │
└────────────────────┴──────────┴──────────────┘



Shows the most recent DuckLake snapshots as proof a new version was created.

In [17]:
print("\nMost recent snapshots:")

con.sql("FROM ducklake_snapshots('lake') ORDER BY snapshot_id DESC LIMIT 5").show()


Most recent snapshots:
┌─────────────┬───────────────────────────────┬────────────────┬───────────────────────────────────────────────────────────────────┬─────────┬────────────────┬───────────────────┐
│ snapshot_id │         snapshot_time         │ schema_version │                              changes                              │ author  │ commit_message │ commit_extra_info │
│    int64    │   timestamp with time zone    │     int64      │                      map(varchar, varchar[])                      │ varchar │    varchar     │      varchar      │
├─────────────┼───────────────────────────────┼────────────────┼───────────────────────────────────────────────────────────────────┼─────────┼────────────────┼───────────────────┤
│           6 │ 2026-08-09 02:30:24.603341+00 │              5 │ {tables_created=[raw.visdrone_frames], tables_inserted_into=[5]}  │ NULL    │ NULL           │ NULL              │
│           5 │ 2026-08-09 02:29:38.660409+00 │              4 │ {tables_ins